# 03 · Record temperatures to CSV

Record a fixed duration to a new CSV. Existing files are refused. Paths are relative to the kernel working directory; serial I/O can extend polling intervals.

Run top to bottom in a **Python 3.12+** kernel. This notebook includes all configuration and setup; no sibling example files are imported. Run only one hardware notebook/process at a time.

**Prepare the filled cooling loop before connecting.** T255P connection sends U, takes serial control and can change AUTO START to RUN, even with target writes disabled. Software disconnect does not stop the pump/chiller or restore its target. Physical hardware has not been validated with this package.

## 1. Environment and imports

Select a kernel with this package and pyserial installed. From the repository root install with `%pip install -e .`; from the examples directory use `%pip install -e ..`. The script-only dashboard (04_dashboard.py) needs the `[gui]` extra, e.g. `%pip install -e "..[gui]"`. Restart the kernel after installation or driver source changes. Imports open no serial ports.

In [1]:
import sys
from pathlib import Path

# Prefer checkout sources when launched from the repository or examples folder.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src/tark_chiller/controller.py").is_file():
        sys.path.insert(0, str(candidate / "src"))
        break

In [2]:
from datetime import UTC, datetime
from math import isfinite
from time import monotonic, sleep

from tark_chiller.controller import T255P, TarkChiller

## 2. Choose the port

This example connects to a **ThermoTek T255P**. Set `PORT` to its serial port (for example `"COM9"`). Adapter metadata identifies the USB adapter, not the attached chiller.

The connection cell below spells out the T255P profile: 9600 baud, 8N1, XON/XOFF, no hardware flow control, and RTS/DTR deasserted. Check that your adapter/cable supports these line levels. Distilled-water target bounds are 5–45 °C, in 0.1 °C steps. See the [hardware guide](../docs/hardware.md) for wiring and the separate CAL protocol.

In [3]:
PORT = "COM9"  # Set the identified serial port, e.g. "COM9".

DURATION_S = 10.0
INTERVAL_S = 1.0

## 3. Connection settings

The next operation cell constructs the driver directly with the settings shown. Construction opens nothing. Its context connects and releases the port, including on a read error or interrupt. Among the notebooks, only example 02 enables target writes; dashboard script 04 also enables explicit GUI target requests.

## 4. Record temperatures to CSV

Record a fixed duration to a new CSV. Existing files are refused. Paths are relative to the kernel working directory; serial I/O can extend polling intervals.

Run the next cell as a unit: it owns connection and cleanup together. Interrupt the cell to cancel; errors remain visible.

In [4]:
if not isinstance(PORT, str) or not PORT.strip():
    raise ValueError('Hardware not configured: set PORT, e.g. "COM9". No port was opened.')

chiller = TarkChiller(
    {
        "port": PORT,
        "baudrate": 9600,
        "bytesize": 8,
        "parity": "N",
        "stopbits": 1,
        "xonxoff": True,
        "rtscts": False,
        "dsrdtr": False,
        "rts": False,
        "dtr": False,
    },
    protocol=T255P(allow_writes=False),
    setpoint_range=(5.0, 45.0),
    coolant_source="ThermoTek T255P Rev C pp9/20; verify installed distilled water",
)

if not isfinite(DURATION_S) or DURATION_S <= 0:
    raise ValueError("DURATION_S must be a positive finite number of seconds")

# 1. Choose a unique UTC filename. Existing files are never overwritten.
stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S_%fZ")
csv_path = Path("outputs") / f"hardware_{stamp}.csv"

with chiller:
    # 2. Start one acquisition worker. It reads temperature, target and status,
    # flushes CSV rows and retains bounded history for snapshot() readers.
    chiller.start_monitoring(interval_s=INTERVAL_S, csv_path=csv_path)
    deadline = monotonic() + DURATION_S
    while monotonic() < deadline:
        snapshot = chiller.snapshot()  # Memory only; no extra device query.
        if snapshot["service_error"] or snapshot["logging_error"] or (not snapshot["running"]):
            raise RuntimeError(
                snapshot["service_error"] or snapshot["logging_error"] or "Monitoring stopped"
            )
        if snapshot["failed_samples"]:
            raise RuntimeError("A hardware poll failed. Inspect the CSV error column and device.")
        sleep(0.1)
    # 3. Finish recording before inspecting the final counters. If the loop
    # raises, the controller context cleans up and preserves that error.
    chiller.stop_monitoring()

    # Catch a failure in the last in-flight poll or while closing the CSV, too.
    snapshot = chiller.snapshot()
    if snapshot["service_error"] or snapshot["logging_error"] or snapshot["failed_samples"]:
        raise RuntimeError(
            snapshot["service_error"] or snapshot["logging_error"] or "A hardware poll failed"
        )
    print(f"Samples: {snapshot['sample_count']}; saved rows: {snapshot['logged_samples']}")
    print(f"CSV saved to: {csv_path.resolve()}")

# Rows include UTC time, elapsed seconds, Celsius readings, status and errors.
# Failed measurements are blank. Ending acquisition does not stop equipment.

Samples: 10; saved rows: 10
CSV saved to: C:\repo\tark-mrc-chiller-control\examples\outputs\hardware_20261021T190252_599108Z.csv


## 5. Check results and finish

Inspect CSV UTC/elapsed times, Celsius readings, status and errors. Failed measurements are blank. A duration-based acquisition does not promise an exact row count.

Context cleanup closes software resources, including after a read error or kernel interrupt. If cleanup fails, resources may remain owned: allow pending I/O to finish and retry `chiller.disconnect(timeout_s=5.0)` if `chiller` was created. Do not rerun configuration over a live connection. A forced kernel termination cannot guarantee cleanup. Follow the equipment's physical shutdown procedure separately.